# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object (not a dictionary)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
# Optionally, print temporal and spatial coverage, keywords, version
print(f"Temporal coverage: {dataset.metadata.temporalCoverage}")
print(f"Spatial coverage: {dataset.metadata.spatialCoverage}")
print(f"Keywords: {dataset.metadata.keywords}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps identify which data entities are available for extraction.

We will print the record set `@id`s defined by the schema.

In [ ]:
# Inspect record sets and their fields
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id} | name: {rs.name}")
    # Print associated fields and columns
    fields = rs.fields
    print("  Fields:")
    for f in fields:
        print(f"    Field @id: {f.id} | name: {f.name} | dataType: {f.data_type}")
        # Print columns if present
        if hasattr(f, 'columns') and f.columns is not None:
            print("      Columns:")
            for c in f.columns:
                print(f"        Column @id: {c.id} | name: {c.name}")

### Sample record inspection
Print a few example records from each record set using their `@id`.

In [ ]:
# Print first 2 records from each record set (referenced by @id)
for rs in record_sets:
    print(f"\nSample records from RecordSet @id: {rs.id}")
    for idx, rec in enumerate(dataset.records(record_set=rs.id)):
        print(rec)
        if idx >= 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all record sets (by @id) into Pandas DataFrames
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Choose the first record set for further analysis
reference_record_set_id = record_set_ids[0]
print(f"DataFrame columns for RecordSet @id '{reference_record_set_id}':")
print(dataframes[reference_record_set_id].columns.tolist())

# Preview the DataFrame
dataframes[reference_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will use a numeric field (by column @id), apply filtering and normalization, and optionally group by a categorical field.

In [ ]:
# Find a numeric column by its @id
df = dataframes[reference_record_set_id]

# For demonstration, select the first available numeric field from the schema overview
numeric_cols = [col for col in df.columns if df[col].dtype != object]
if numeric_cols:
    numeric_field_id = numeric_cols[0]  # Use column @id
    print(f"Using numeric column: {numeric_field_id}")
    
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by a categorical field (use first object dtype column)
    group_fields = [col for col in df.columns if df[col].dtype == object]
    if group_fields:
        group_field_id = group_fields[0]
        print(f"Grouping by column @id: {group_field_id}")
        grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(grouped.head())
else:
    print("No numeric columns found in DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We use matplotlib for basic plots.

If the dataset has numeric and categorical fields, visualize their relationship.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols and group_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("No suitable numeric and group field found for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the FAIR² dataset referenced by Croissant schema at [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

- We reviewed dataset metadata and record sets, referencing entities by their `@id`.
- Extracted records from each record set and performed exploratory data analysis using their column `@id`s.
- Applied filtering, normalization, and grouping on numeric and categorical fields.
- Generated visualizations for numeric distributions and relationships.

For more advanced exploration and analysis, refer to the [mlcroissant documentation](https://mlcroissant.readthedocs.io/).